# Matemáticas de la Inteligencia Artificial
## Sesión 15 — Generar, experimentar y reconstruir el modelo completo

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CuentosCuanticos/matematicas-ia/blob/main/15_generacion_mini_gpt/laboratorio.ipynb)

**Objetivo.** Cerrar el curso recorriendo

\[
\text{prompt}\to\text{tokens}\to\text{Transformer}\to\text{logits}
\to p(t_{n+1}\mid t_{\le n})\to\text{sampling}\to t_{n+1}.
\]

Trabajaremos con un mini-Transformer causal entrenado sobre un corpus diminuto y controlado. El interés no es la calidad literaria, sino experimentar con **greedy, temperatura, top-k, top-p, ventana de contexto y KV cache**, y separar probabilidad lingüística de verdad.


In [ ]:
import math, random
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED=15
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("dispositivo:",device)

frases=[
"la masa curva el espacio tiempo .",
"la luz sigue geodesicas nulas .",
"un modelo de lenguaje predice el siguiente token .",
"la atencion compara queries y keys .",
"los values transportan informacion .",
"la temperatura modifica la distribucion de muestreo .",
"una probabilidad alta no garantiza verdad .",
"el contexto condiciona la prediccion .",
"la kv cache reutiliza keys y values anteriores .",
"el entrenamiento modifica parametros .",
"el prompt modifica el contexto pero no los pesos .",
"el fine tuning modifica los parametros .",
"top k conserva los k tokens mas probables .",
"top p conserva una masa acumulada de probabilidad .",
"la softmax normaliza logits en una distribucion .",
"el gradiente indica como cambia la perdida .",
"backpropagation aplica la regla de la cadena .",
"un transformer combina atencion y mlp .",
"la mascara causal impide mirar al futuro .",
"el modelo no tiene una funcion de verdad ."]
tok=(" ".join(frases*35)).split()
vocab=sorted(set(tok)); stoi={t:i for i,t in enumerate(vocab)}; itos={i:t for t,i in stoi.items()}
data=torch.tensor([stoi[t] for t in tok],dtype=torch.long)

def encode(s): return [stoi[t] for t in s.split()]
def decode(ids): return " ".join(itos[int(i)] for i in ids)
print("tokens:",len(data),"| vocabulario:",len(vocab))


In [ ]:
class Attn(nn.Module):
    def __init__(self,d,h):
        super().__init__(); assert d%h==0
        self.h=h; self.dh=d//h
        self.qkv=nn.Linear(d,3*d,bias=False); self.proj=nn.Linear(d,d,bias=False)
    def forward(self,x):
        B,T,C=x.shape
        q,k,v=self.qkv(x).chunk(3,-1)
        q=q.view(B,T,self.h,self.dh).transpose(1,2)
        k=k.view(B,T,self.h,self.dh).transpose(1,2)
        v=v.view(B,T,self.h,self.dh).transpose(1,2)
        s=(q@k.transpose(-2,-1))/math.sqrt(self.dh)
        mask=torch.triu(torch.ones(T,T,dtype=torch.bool,device=x.device),1)
        A=F.softmax(s.masked_fill(mask,float("-inf")),-1)
        z=(A@v).transpose(1,2).contiguous().view(B,T,C)
        return self.proj(z)

class Block(nn.Module):
    def __init__(self,d=48,h=4,ff=128):
        super().__init__(); self.n1=nn.LayerNorm(d); self.a=Attn(d,h); self.n2=nn.LayerNorm(d)
        self.m=nn.Sequential(nn.Linear(d,ff),nn.GELU(),nn.Linear(ff,d))
    def forward(self,x):
        x=x+self.a(self.n1(x)); return x+self.m(self.n2(x))

class MiniGPT(nn.Module):
    def __init__(self,V,C=16,d=48,L=2):
        super().__init__(); self.C=C
        self.e=nn.Embedding(V,d); self.p=nn.Embedding(C,d)
        self.b=nn.ModuleList([Block(d) for _ in range(L)])
        self.n=nn.LayerNorm(d); self.head=nn.Linear(d,V,bias=False)
    def forward(self,idx,y=None):
        B,T=idx.shape
        if T>self.C: raise ValueError("contexto demasiado largo")
        x=self.e(idx)+self.p(torch.arange(T,device=idx.device))[None]
        for b in self.b: x=b(x)
        z=self.head(self.n(x)); loss=None
        if y is not None: loss=F.cross_entropy(z.reshape(-1,z.size(-1)),y.reshape(-1))
        return z,loss

model=MiniGPT(len(vocab)).to(device)
C=model.C
def batch(B=32):
    s=torch.randint(0,len(data)-C-1,(B,))
    x=torch.stack([data[i:i+C] for i in s]); y=torch.stack([data[i+1:i+C+1] for i in s])
    return x.to(device),y.to(device)

opt=torch.optim.AdamW(model.parameters(),lr=3e-3)
losses=[]
model.train()
for step in range(70):
    x,y=batch(); _,loss=model(x,y); opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
model.eval()
print("parámetros:",sum(p.numel() for p in model.parameters()),"| loss final:",losses[-1])

def logits_next(ids):
    x=torch.tensor(ids[-C:],device=device)[None]
    with torch.no_grad(): z,_=model(x)
    return z[0,-1]


## 1. Decodificación

Con temperatura \(T>0\),

\[
p_i(T)=\frac{e^{z_i/T}}{\sum_j e^{z_j/T}}.
\]

`top-k` conserva \(k\) candidatos; `top-p` conserva el conjunto mínimo cuya masa acumulada supera \(p\). Completa las funciones y después experimenta con la **misma red y los mismos pesos**.


In [ ]:
def temperature(z,T=1.0):
    # TODO 1
    return ...

def topk(z,k=None):
    if k is None: return z
    # TODO 2: deja -inf fuera de los k logits mayores.
    return ...

def topp(z,p=None):
    if p is None: return z
    # TODO 3: ordena, calcula masa acumulada y conserva el núcleo.
    return ...

def choose(z,T=1.0,k=None,p=None,greedy=False,g=None):
    if greedy: return int(torch.argmax(z))
    # TODO 4: temperatura -> top-k -> top-p -> softmax -> multinomial
    return ...

def generate(prompt,n=20,seed=0,**cfg):
    ids=encode(prompt); g=torch.Generator(device=device); g.manual_seed(seed); lp=[]
    for _ in range(n):
        z=logits_next(ids)
        # TODO 5: construye la distribución usada para el muestreo.
        zz=...
        probs=...
        nxt=choose(z,g=g,**cfg)
        lp.append(float(torch.log(probs[nxt]+1e-12))); ids.append(nxt)
    return ids,lp


In [ ]:
configs={"greedy":dict(greedy=True),"T=.6":dict(T=.6),"T=1":dict(T=1.0),
         "top-k":dict(T=.9,k=5),"top-p":dict(T=.9,p=.8)}
for name,cfg in configs.items():
    ids,_=generate("un modelo de lenguaje",18,7,**cfg)
    print(f"\n{name:7s} -> {decode(ids)}")


## 2. Entropía, diversidad y contexto

La entropía categórica es

\[
H(p)=-\sum_i p_i\log p_i.
\]

Además mediremos `distinct-1`. Después comprobaremos que el contexto usado por el modelo nunca supera \(C\).


In [ ]:
def entropy(pr):
    # TODO 6
    return ...

def distinct1(ids):
    # TODO 7
    return ...

z=logits_next(encode("la temperatura modifica"))
for T in [.4,.7,1.,1.5,2.]:
    pr=F.softmax(temperature(z,T),-1)
    print(T,entropy(pr))

larga=encode("el prompt modifica el contexto pero no los pesos la temperatura modifica la distribucion de muestreo una probabilidad alta no garantiza verdad el modelo no tiene una funcion de verdad")
# TODO 8: ventana exacta que recibe el modelo
ventana=...
print("total:",len(larga),"| ventana:",decode(ventana))


## 3. KV cache

La ventana determina **qué información cabe**. La KV cache solo evita recalcular \(K\) y \(V\) antiguos. Verifica en una cabeza que la salida de la última posición coincide usando cálculo completo o cacheado.


In [ ]:
torch.manual_seed(123)
T,d=6,8; X=torch.randn(T,d); Wq=torch.randn(d,d); Wk=torch.randn(d,d); Wv=torch.randn(d,d)
Q=X@Wq; K=X@Wk; V=X@Wv

def full_last(Q,K,V):
    q=Q[-1:]
    # TODO 9
    return ...

def cached_last(q,Kc,Vc,k,v):
    # TODO 10
    return ...

a=full_last(Q,K,V); b=cached_last(Q[-1:],K[:-1],V[:-1],K[-1:],V[-1:])
print("diferencia:",(a-b).abs().max().item())
assert torch.allclose(a,b,atol=1e-6)


## 4. Pesos, contexto y verdad

Durante inferencia, cambiar el prompt cambia \(p_\theta(\cdot\mid P)\), pero no cambia \(\theta\). Y una probabilidad alta de token **no es** una probabilidad de verdad.


In [ ]:
checksum=lambda: sum(float(p.detach().double().sum()) for p in model.parameters())
a=checksum()
p1=F.softmax(logits_next(encode("el prompt modifica")),-1)
p2=F.softmax(logits_next(encode("la softmax normaliza")),-1)
b=checksum()
print("pesos iguales:",a==b,"| distancia L1 entre distribuciones:",float((p1-p2).abs().sum()))

for prompt in ["una probabilidad alta","una probabilidad alta no garantiza"]:
    pr=F.softmax(logits_next(encode(prompt)),-1); v,i=torch.topk(pr,5)
    print("\n",prompt,[(itos[int(j)],round(float(x),3)) for x,j in zip(v,i)],"suma=",float(pr.sum()))


# Problema final abierto — Un generador científico mínimo

Diseña tres modos:

- **preciso**: alta plausibilidad y poca variación;
- **exploratorio**: más diversidad sin aproximarse a muestreo uniforme;
- **prudente**: para preguntas factuales solo responde si existe evidencia explícita.

Debes:

1. justificar \(T\), `top-k` y/o `top-p`;
2. evaluar cinco prompts con log-probabilidad media, `distinct-1` y repetición de bigramas;
3. usar esta evidencia externa:

```python
evidence={
 "probabilidad":"una probabilidad alta no garantiza verdad",
 "kv cache":"la kv cache reutiliza keys y values anteriores",
 "prompt":"el prompt modifica el contexto pero no los pesos"}
```

4. si no hay evidencia, `prudente` debe devolver una abstención;
5. explicar qué parte pertenece al modelo y qué parte al sistema diseñado;
6. explicar por qué modificar prompt/decodificación no es *fine-tuning*.

No hay una única configuración correcta: se evalúa la justificación matemática.


In [ ]:
# TODO 11: repetición de bigramas
def rep2(ids): return ...

# TODO 12: evaluar una configuración
def evaluate(prompt,cfg,n=20,seed=0): return ...

evidence={"probabilidad":"una probabilidad alta no garantiza verdad",
          "kv cache":"la kv cache reutiliza keys y values anteriores",
          "prompt":"el prompt modifica el contexto pero no los pesos"}

# TODO 13: respuesta prudente con abstención si falta evidencia
def cautious(topic): return ...

# TODO 14: tus configuraciones justificadas
configs_finales={"preciso":{},"exploratorio":{}}


## Cierre

\[
\boxed{\text{ventana de contexto}\neq\text{KV cache}},\qquad
\boxed{\text{prompt}\neq\text{fine-tuning}},\qquad
\boxed{\text{probabilidad lingüística}\neq\text{verdad}}.
\]

El curso termina pudiendo reconstruir y modificar conscientemente la cadena completa desde los datos y los parámetros hasta la generación del siguiente token.
